# 🧪 CASMI 2026: Fast Validation & Comparative Benchmark Harness

> **TL;DR** — Submitting to the Kaggle public leaderboard takes significant time, and the public test set is composed of synthetic `enveda-180` rows (the wrong chemistry!). 
> This notebook provides an **honest, leak-free local validation pipeline** using the 1,145-molecule validation dataset (`val.parquet`, `val_key.pkl`, `scaffolds.pkl`).
>
> ### 📊 Validation Cohorts:
> 1. **`C1_npx` (245 molecules)**: True natural products from `enveda-np-examples` on timsTOF (the exact target chemistry and instrument profile).
> 2. **`C1_xlib` (300 molecules)**: Cross-library natural products with reference spectra in public repositories (MassBank, GNPS, RIKEN).
> 3. **`C2` (400 molecules)**: Realistic Class 2 natural products with **NO reference spectra** (self-match banned).
> 4. **`C2_hard` (200 molecules)**: Class 2 natural products with **scaffold holdout** (all related Murcko scaffolds banned).
>
> ### 🚀 What This Harness Evaluates:
> - **Level 1 (Ultra-Fast ~15s)**: Channel 1 coverage and Candidate Pool recall deltas (testing offset centering and window expansion).
> - **Level 2 (Full Pipeline ~2m)**: Side-by-side **Baseline vs. Phase 1 / Phase 2 Enhanced** evaluation computing exact **MRR@25**, **Top-1 Accuracy**, and **Hit@25** per cohort.


In [ ]:
# ===================================================================================
#  Environment Setup & Dataset Discovery
# ===================================================================================
import os, sys, glob, time, pickle, math, subprocess, zipfile
from multiprocessing import Pool as MPool
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import pyarrow as pa
import matplotlib.pyplot as plt

T0 = time.time()

def find_file(name):
    hits = glob.glob(f'/kaggle/input/**/{name}', recursive=True)
    if not hits:
        hits = glob.glob(f'./**/{name}', recursive=True)
    if not hits:
        raise FileNotFoundError(f"File not found in /kaggle/input or current dir: {name}")
    return sorted(hits, key=len)[0]

# 1. Locate Competition Files
COMP   = os.path.dirname(find_file('train.parquet'))
TRAIN  = os.path.join(COMP, 'train.parquet')
SAMPLE = os.path.join(COMP, 'sample_submission.csv')

# 2. Locate / Extract Validation Dataset (val_scripts.zip or val.parquet)
val_dir = '/kaggle/working/val'
os.makedirs(val_dir, exist_ok=True)

try:
    val_zip = find_file('val_scripts.zip')
    print(f"[INFO] Found validation archive: {val_zip}. Extracting to {val_dir}...")
    with zipfile.ZipFile(val_zip, 'r') as z:
        z.extractall(val_dir)
except Exception:
    print("[INFO] val_scripts.zip not found as archive, searching for unzipped files...")

VAL_PARQUET = find_file('val.parquet')
VAL_KEY     = find_file('val_key.pkl')

print(f"[SUCCESS] Validation files located:")
print(f"  val.parquet: {VAL_PARQUET} ({os.path.getsize(VAL_PARQUET)/1e6:.2f} MB)")
print(f"  val_key.pkl: {VAL_KEY} ({os.path.getsize(VAL_KEY)/1e6:.2f} MB)")

# 3. Bootstrap offline RDKit
rdkit_whls = glob.glob('/kaggle/input/**/rdkit-*.whl', recursive=True) + glob.glob('./**/rdkit-*.whl', recursive=True)
if rdkit_whls:
    print(f"[INFO] Installing offline RDKit wheel: {rdkit_whls[0]}")
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-index', rdkit_whls[0]], check=False)

try:
    from rdkit import Chem, RDLogger
    from rdkit.Chem import rdFingerprintGenerator, MACCSkeys
    from rdkit.Chem.Descriptors import ExactMolWt
    RDLogger.DisableLog('rdApp.*')
    HAVE_RDKIT = True
    print("[SUCCESS] RDKit is active and verified.")
except Exception as e:
    HAVE_RDKIT = False
    print(f"[WARNING] RDKit unavailable ({e}).")


In [ ]:
# ===================================================================================
#  CONFIG & Fast Evaluation Parameters
# ===================================================================================
class CFG:
    # Baseline Parameters
    PPM_WIN_BASE   = 8.5
    PPM_FALLBACK   = 30.0

    # Phase 1 Enhanced Parameters
    PPM_OFFSET     = 1.45   # Measured +1.45 ppm systematic timsTOF mass offset
    PPM_EXPAND     = (8.5, 20.0, 45.0, 80.0) # Multi-tier progressive window

    # Preprocessing
    INT_FLOOR      = 0.002
    MAX_PEAKS      = 256
    MZ_TOL         = 0.01
    INT_POWER      = 1.0
    ENT_WEIGHT     = True

    # Channel 2 Analog
    ANALOG_WIN     = 200.0
    N_ANALOG       = 100
    SIM_POWER      = 4.0

    # Model / Ranker
    W1_PRIORS      = (0.30, 0.60)
    SEEDS          = (0, 1, 2, 3)
    GBM = dict(max_depth=6, max_iter=500, learning_rate=0.03,
               min_samples_leaf=80, l2_regularization=1.0)
    TOPN           = 25


In [ ]:
# ===================================================================================
#  Numba-Accelerated Spectral Processing & Search Functions
# ===================================================================================
import numpy as np
from numba import njit, prange

@njit(cache=True, fastmath=True)
def _clean(mz, it, floor, topk, power, ent_weight):
    n = len(mz)
    if n == 0: return np.empty(0, np.float32), np.empty(0, np.float32)
    mx = 0.0
    for i in range(n):
        if it[i] > mx: mx = it[i]
    if mx <= 0: return np.empty(0, np.float32), np.empty(0, np.float32)
    thr = floor * mx
    c = 0
    for i in range(n):
        if it[i] >= thr: c += 1
    idx = np.empty(c, np.int64)
    j = 0
    for i in range(n):
        if it[i] >= thr:
            idx[j] = i; j += 1
    if c > topk:
        v = np.empty(c, np.float32)
        for i in range(c): v[i] = it[idx[i]]
        o = np.argsort(v)[c - topk:]
        k2 = np.empty(topk, np.int64)
        for i in range(topk): k2[i] = idx[o[i]]
        k2.sort()
        idx = k2
        c = topk
    om = np.empty(c, np.float32)
    oi = np.empty(c, np.float32)
    s = 0.0
    for i in range(c):
        om[i] = mz[idx[i]]
        v = it[idx[i]] ** power
        oi[i] = v
        s += v
    if s > 0:
        for i in range(c): oi[i] /= s
    if ent_weight:
        S = 0.0
        for i in range(c):
            if oi[i] > 0: S -= oi[i] * np.log(oi[i])
        if S < 3.0:
            w = 0.25 + 0.25 * S
            s2 = 0.0
            for i in range(c):
                oi[i] = oi[i] ** w
                s2 += oi[i]
            if s2 > 0:
                for i in range(c): oi[i] /= s2
    return om, oi

@njit(cache=True, fastmath=True)
def entropy_sim(qmz, qp, cmz, cp, tol):
    i = 0; j = 0; n = len(qmz); m = len(cmz)
    SA = 0.0
    for x in range(n):
        if qp[x] > 0: SA -= qp[x] * np.log(qp[x])
    SB = 0.0
    for x in range(m):
        if cp[x] > 0: SB -= cp[x] * np.log(cp[x])
    SAB = 0.0; tot = 0.0
    buf = np.empty(n + m, np.float64); b = 0
    while i < n and j < m:
        d = qmz[i] - cmz[j]
        if d < -tol:
            buf[b] = qp[i]; i += 1; b += 1
        elif d > tol:
            buf[b] = cp[j]; j += 1; b += 1
        else:
            buf[b] = qp[i] + cp[j]; i += 1; j += 1; b += 1
    while i < n: buf[b] = qp[i]; i += 1; b += 1
    while j < m: buf[b] = cp[j]; j += 1; b += 1
    for x in range(b): tot += buf[x]
    if tot <= 0: return 0.0
    for x in range(b):
        v = buf[x] / tot
        if v > 0: SAB -= v * np.log(v)
    return 1.0 - (2.0 * SAB - SA - SB) / np.log(4.0)

@njit(cache=True, fastmath=True)
def entropy_sim_shift(qmz, qp, cmz, cp, tol, shift):
    """Evaluates joint direct and mass-shifted spectral entropy similarity."""
    a = entropy_sim(qmz, qp, cmz, cp, tol)
    if shift > -0.001 and shift < 0.001:
        return a
    sm = np.empty(len(cmz), np.float32)
    for i in range(len(cmz)):
        sm[i] = cmz[i] + shift
    b = entropy_sim(qmz, qp, sm, cp, tol)
    return a if a > b else b

@njit(cache=True, fastmath=True, parallel=True)
def search(qmz, qp, cand, off, allmz, allin, tol, floor, topk, power, ent_weight):
    out = np.zeros(len(cand), np.float32)
    for k in prange(len(cand)):
        c = cand[k]; a = off[c]; b = off[c + 1]
        if b <= a: continue
        cm, cp = _clean(allmz[a:b], allin[a:b], floor, topk, power, ent_weight)
        if len(cm) == 0: continue
        out[k] = entropy_sim(qmz, qp, cm, cp, tol)
    return out

@njit(cache=True, fastmath=True, parallel=True)
def search_shift(qmz, qp, cand, off, allmz, allin, tol, floor, topk, power, ent_weight, shift):
    out = np.zeros(len(cand), np.float32)
    for k in prange(len(cand)):
        c = cand[k]; a = off[c]; b = off[c + 1]
        if b <= a: continue
        cm, cp = _clean(allmz[a:b], allin[a:b], floor, topk, power, ent_weight)
        if len(cm) == 0: continue
        out[k] = entropy_sim_shift(qmz, qp, cm, cp, tol, shift[k])
    return out


In [ ]:
# ===================================================================================
#  Physics, Leak-Free Reference Library & Candidate Pool
# ===================================================================================
MASS = dict(C=12.0, H=1.00782503207, N=14.0030740048, O=15.9949146196, P=30.97376163,
            S=31.97207100, F=18.99840322, Cl=34.96885268, Br=78.9183371, I=126.904473,
            Na=22.9897692809, K=38.96370668, Si=27.9769265325, B=11.0093054, Se=79.9165213)
E = 0.00054857990
PROTON = MASS['H'] - E
H2O = 2 * MASS['H'] + MASS['O']
NH4 = MASS['N'] + 4 * MASS['H']
FORMATE = MASS['C'] + 2 * MASS['H'] + 2 * MASS['O']
ACETATE = 2 * MASS['C'] + 4 * MASS['H'] + 2 * MASS['O']

ADDUCTS = {
    "[M+H]+": (1, 1, PROTON), "[M+NH4]+": (1, 1, NH4 - E), "[M+Na]+": (1, 1, MASS['Na'] - E),
    "[M+K]+": (1, 1, MASS['K'] - E), "[M-H2O+H]+": (1, 1, PROTON - H2O), "[M-2H2O+H]+": (1, 1, PROTON - 2*H2O),
    "[M+2H]2+": (1, 2, 2 * PROTON), "[M]+": (1, 1, -E), "[M-H2O]+": (1, 1, -E - H2O),
    "[M+CH3OH+H]+": (1, 1, PROTON + MASS['C'] + 4 * MASS['H'] + MASS['O']),
    "[M+CH3CN+H]+": (1, 1, PROTON + 2 * MASS['C'] + 3 * MASS['H'] + MASS['N']),
    "[M-H]-": (1, 1, -PROTON), "[M-H2O-H]-": (1, 1, -PROTON - H2O), "[M+CH2O2-H]-": (1, 1, FORMATE - PROTON),
    "[M+C2H4O2-H]-": (1, 1, ACETATE - PROTON), "[M+Cl]-": (1, 1, MASS['Cl'] + E), "[M]-": (1, 1, E),
    "[M-2H]-": (1, 2, -2 * PROTON), "[M+Na-2H]-": (1, 1, MASS['Na'] - 2 * PROTON),
    "[2M+H]+": (2, 1, PROTON), "[2M+Na]+": (2, 1, MASS['Na'] - E), "[2M+NH4]+": (2, 1, NH4 - E),
    "[2M+K]+": (2, 1, MASS['K'] - E), "[2M-H]-": (2, 1, -PROTON), "[2M+CH2O2-H]-": (2, 1, FORMATE - PROTON),
    "[2M+C2H4O2-H]-": (2, 1, ACETATE - PROTON), "[2M+Na-2H]-": (2, 1, MASS['Na'] - 2 * PROTON),
    "[3M+H]+": (3, 1, PROTON), "[3M-H]-": (3, 1, -PROTON),
}

def neutral_mass(mz, adduct):
    out = np.full(len(mz), np.nan)
    ad = np.asarray(adduct, dtype=object)
    for a, (n, z, d) in ADDUCTS.items():
        m = (ad == a)
        if m.any():
            out[m] = (mz[m] * z - d) / n
    return out

def _drop(cand, ban_rows, ban_keys, ik_arr):
    """Prevents data leakage by removing banned training rows and banned scaffold keys."""
    if cand.size == 0 or (len(ban_rows) == 0 and len(ban_keys) == 0):
        return cand
    m = np.ones(len(cand), bool)
    if len(ban_rows):
        m &= ~np.isin(cand, ban_rows)
    if len(ban_keys):
        m &= ~np.isin(ik_arr[cand], ban_keys)
    return cand[m]

def load_library(path):
    t0 = time.time()
    t = pq.read_table(path, columns=['inchikey14', 'normalized_smiles', 'adduct', 'ionization_mode',
                                     'precursor_mz', 'ms2_mzs', 'ms2_normalized_intensities'])
    mzc = t.column('ms2_mzs').combine_chunks()
    itc = t.column('ms2_normalized_intensities').combine_chunks()
    off = mzc.offsets.to_numpy().astype(np.int64)
    allmz = mzc.values.to_numpy(zero_copy_only=False).astype(np.float32)
    allin = itc.values.to_numpy(zero_copy_only=False).astype(np.float32)
    prec = t.column('precursor_mz').to_numpy(zero_copy_only=False).astype(np.float64)
    add = np.asarray(t.column('adduct').cast(pa.string()).to_pylist(), dtype=object)
    ik  = np.asarray(t.column('inchikey14').cast(pa.string()).to_pylist(), dtype=object)
    smi = np.asarray(t.column('normalized_smiles').cast(pa.string()).to_pylist(), dtype=object)
    
    imode = np.asarray(t.column('ionization_mode').cast(pa.string()).to_pylist(), dtype=object)
    is_pos = np.array([str(m).lower() == 'positive' for m in imode], dtype=bool)

    nm  = neutral_mass(prec, add)
    ok  = np.isfinite(nm)
    order = np.argsort(np.where(ok, nm, 1e18), kind='mergesort')
    best = {}
    for k, s in zip(ik, smi):
        if k and s and k not in best:
            best[k] = s
    print(f"[INFO] Reference Library loaded: {len(off)-1:,} spectra / {len(best):,} unique structures ({time.time()-t0:.1f}s)", flush=True)
    return dict(off=off, mz=allmz, it=allin, nm=nm, ik=ik, is_pos=is_pos, best=best,
                order=order, snm=nm[order], n_ok=int(ok.sum()))

def lib_window(L, target, tol):
    lo = np.searchsorted(L['snm'][:L['n_ok']], target - tol, 'left')
    hi = np.searchsorted(L['snm'][:L['n_ok']], target + tol, 'right')
    return L['order'][lo:hi]

def build_rep_polarity(L):
    """Polarity-aware representative scaffold library."""
    npk = np.diff(L['off'])
    best = {}; ik = L['ik']; is_pos = L['is_pos']
    for i in range(len(ik)):
        k = ik[i]
        if k:
            rep_key_tuple = (k, bool(is_pos[i]))
            if rep_key_tuple not in best or npk[i] > npk[best[rep_key_tuple]]:
                best[rep_key_tuple] = i
    rep_idx = np.array(sorted(best.values()))
    nm = L['nm'][rep_idx]
    ok = np.isfinite(nm)
    rep_idx = rep_idx[ok]; nm = nm[ok]
    rep_key = ik[rep_idx]; rep_is_pos = is_pos[rep_idx]
    o = np.argsort(nm)
    print(f"[INFO] Built polarity-aware representative analog set: {len(rep_idx):,} spectra across {len(set(rep_key)):,} unique scaffolds.", flush=True)
    return rep_idx[o], rep_key[o], nm[o], rep_is_pos[o]

# Candidate Pool Setup
BITS = np.load(find_file('fp_bits.npy'))
_g = {}

def _fp_init():
    if not HAVE_RDKIT: return
    _g['m2'] = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=4096)
    _g['m3'] = rdFingerprintGenerator.GetMorganGenerator(radius=3, fpSize=4096)
    _g['rk'] = rdFingerprintGenerator.GetRDKitFPGenerator(fpSize=2048, maxPath=6)

def fp_and_mass(smi):
    if not HAVE_RDKIT: return None
    if not _g: _fp_init()
    m = Chem.MolFromSmiles(smi)
    if m is None: return None
    try:
        fp = np.concatenate([
            _g['m2'].GetFingerprintAsNumPy(m).astype(np.uint8),
            _g['m3'].GetFingerprintAsNumPy(m).astype(np.uint8),
            _g['rk'].GetFingerprintAsNumPy(m).astype(np.uint8),
            np.array(MACCSkeys.GenMACCSKeys(m), dtype=np.uint8)
        ])[BITS]
        return fp, float(ExactMolWt(m))
    except Exception:
        return None

class CandidatePool:
    def __init__(self, fp, mass, keys, smiles, nbits):
        o = np.argsort(mass)
        self._fp = fp[o]
        self.mass = mass[o]
        self.keys = np.asarray(keys, dtype=object)[o]
        self.smiles = np.asarray(smiles, dtype=object)[o]
        self.nbits = nbits
        self.k2i = {k: i for i, k in enumerate(self.keys)}

    def window(self, t, ppm):
        a = np.searchsorted(self.mass, t * (1 - ppm / 1e6), 'left')
        b = np.searchsorted(self.mass, t * (1 + ppm / 1e6), 'right')
        return np.arange(a, b)

    def fps(self, idx):
        return np.unpackbits(np.asarray(self._fp[idx]), axis=1)[:, :self.nbits]

def build_candidate_pool():
    t0 = time.time()
    d = os.path.dirname(find_file('coco_fp.npy'))
    cm = pickle.load(open(os.path.join(d, 'coco_meta.pkl'), 'rb'))
    co_fp = np.load(os.path.join(d, 'coco_fp.npy'))
    co_mass = np.load(os.path.join(d, 'coco_mass.npy'))
    co_keys = np.asarray(cm['keys'], dtype=object)
    co_smis = np.asarray(cm['smiles'], dtype=object)
    
    tr = pq.read_table(TRAIN, columns=['inchikey14', 'normalized_smiles']).to_pandas()
    tr = tr.dropna().drop_duplicates('inchikey14')
    coco_keys_set = set(co_keys)
    tr = tr[~tr.inchikey14.isin(coco_keys_set)]
    
    if HAVE_RDKIT:
        with MPool(4) as mp:
            res = mp.map(fp_and_mass, list(tr.normalized_smiles), chunksize=500)
        ok = [i for i, r in enumerate(res) if r is not None]
        tr_fp = np.packbits(np.stack([res[i][0] for i in ok]), axis=1)
        tr_mass = np.array([res[i][1] for i in ok])
        tr_keys = tr.inchikey14.values[ok]
        tr_smi = tr.normalized_smiles.values[ok]
    else:
        tr_fp = np.empty((0, co_fp.shape[1]), dtype=np.uint8)
        tr_mass = np.empty(0, dtype=np.float64)
        tr_keys = np.empty(0, dtype=object)
        tr_smi = np.empty(0, dtype=object)

    fp = np.vstack([co_fp, tr_fp])
    mass = np.concatenate([co_mass, tr_mass])
    keys = np.concatenate([co_keys, tr_keys])
    smis = np.concatenate([co_smis, tr_smi])
    good = np.isfinite(mass)

    pool = CandidatePool(fp[good], mass[good], keys[good], smis[good], cm['nbits'])
    print(f"[SUCCESS] Unified candidate pool ready: {len(pool.mass):,} structures ({time.time()-t0:.1f}s)", flush=True)
    return pool


In [ ]:
# ===================================================================================
#  Load Validation Ground Truth & Leakage Banning Sets
# ===================================================================================
key_list = pickle.load(open(VAL_KEY, 'rb'))
TRUTH  = {k['molecule_id']: k['inchikey14'] for k in key_list}
COHORT = {k['molecule_id']: k['cohort']     for k in key_list}
BAN    = {k['molecule_id']: (np.asarray(k['ban_rows'], np.int64),
                              np.asarray(k['ban_keys'], dtype=object)) for k in key_list}

val_df = pq.read_table(VAL_PARQUET).to_pandas()
val_df['nm'] = neutral_mass(val_df.precursor_mz.values.astype(np.float64), val_df.adduct.values)
val_mols = list(val_df.groupby('molecule_id'))

print(f"[INFO] Validation cohort loaded: {len(val_mols):,} molecules")
print("Cohort distribution:")
print(pd.Series(COHORT).value_counts().to_string())


In [ ]:
# ===================================================================================
#  LEVEL 1 BENCHMARK: Quick Candidate Pool Recall & Channel 1 Coverage (~15s)
# ===================================================================================
L = load_library(TRAIN)
pool = build_candidate_pool()

def quick_candidate_benchmark(val_mols, calibrate_offset=False, ppm_tiers=(8.5, 30.0)):
    """Evaluates Channel 1 and Candidate Pool recall across the 1,145 validation molecules."""
    rows = []
    for mid, sub in val_mols:
        nms = sub.nm.values[np.isfinite(sub.nm.values)]
        if len(nms) == 0:
            rows.append(dict(mid=mid, cohort=COHORT[mid], lv_hit=False, pool_hit=False, n_cand=0))
            continue

        raw_target = float(np.median(nms))
        target = raw_target * (1.0 - 1.45e-6) if calibrate_offset else raw_target
        specs  = [(r.ms2_mzs, r.ms2_normalized_intensities) for r in sub.itertuples()]
        br, bk = BAN[mid]
        truth  = TRUTH[mid]

        # Channel 1: Candidate Library Search with Leakage Banning
        cand_lib = lib_window(L, target, target * 8.5 / 1e6)
        cand_lib = _drop(cand_lib, br, bk, L['ik'])

        lv_hit = False
        if len(cand_lib):
            best = -1.0
            for mz, it in specs:
                qm, qp = clean_spectrum(mz, it)
                if len(qm) == 0: continue
                sc = search(qm, qp, cand_lib, L['off'], L['mz'], L['it'],
                            CFG.MZ_TOL, CFG.INT_FLOOR, CFG.MAX_PEAKS, CFG.INT_POWER, CFG.ENT_WEIGHT)
                for c, s in zip(cand_lib, sc):
                    if L['ik'][c] == truth and s > best:
                        best = s
            if best >= 0:
                lv_hit = True

        # Candidate Pool Window Expansion
        cand = pool.window(target, ppm_tiers[0])
        for p in ppm_tiers[1:]:
            if len(cand) >= CFG.TOPN: break
            cand = pool.window(target, p)
        pool_hit = truth in set(pool.keys[cand]) if len(cand) else False

        rows.append(dict(mid=mid, cohort=COHORT[mid], lv_hit=lv_hit, pool_hit=pool_hit, n_cand=len(cand)))
    return pd.DataFrame(rows)

t0 = time.time()
print("[INFO] Running Baseline Config (Uncalibrated, 8.5 -> 30 ppm fallback on zero)...")
df_base = quick_candidate_benchmark(val_mols, calibrate_offset=False, ppm_tiers=(8.5, 30.0))

print("[INFO] Running Phase 1 Config (+1.45 ppm Centering, Progressive Tiers: 8.5 -> 20 -> 45 -> 80 ppm)...")
df_p1 = quick_candidate_benchmark(val_mols, calibrate_offset=True, ppm_tiers=(8.5, 20.0, 45.0, 80.0))
print(f"[SUCCESS] Level 1 benchmark completed in {time.time()-t0:.1f}s!")

# Compute Level 1 Deltas
base_grouped = df_base.groupby('cohort').agg(
    pool_recall=('pool_hit', 'mean'),
    lv_recall=('lv_hit', 'mean'),
    mean_cand=('n_cand', 'mean')
)
p1_grouped = df_p1.groupby('cohort').agg(
    pool_recall=('pool_hit', 'mean'),
    lv_recall=('lv_hit', 'mean'),
    mean_cand=('n_cand', 'mean')
)

delta_df = pd.DataFrame({
    'Base Pool Recall': base_grouped['pool_recall'],
    'Phase 1 Pool Recall': p1_grouped['pool_recall'],
    'Delta Pool Recall': p1_grouped['pool_recall'] - base_grouped['pool_recall'],
    'Base Cand Count': base_grouped['mean_cand'].round(1),
    'Phase 1 Cand Count': p1_grouped['mean_cand'].round(1),
})
print("\n--- LEVEL 1 VALIDATION SCORECARD (Candidate Pool Reachability) ---")
print(delta_df.to_string())


In [ ]:
# ===================================================================================
#  LEVEL 2 BENCHMARK: Full End-to-End MRR@25 Evaluation
# ===================================================================================
# Load Neural Models & Ranker Weights
_MODEL = None
load_neural_models()
rank_train_path = find_file('rank_train.npz')
z = np.load(rank_train_path)
NFEAT = z['X'].shape[1]

RANKERS = []
for w1 in CFG.W1_PRIORS:
    W = np.where(z['M'] == 0, w1, 1.0 - w1)
    for sd in CFG.SEEDS:
        m = HistGradientBoostingClassifier(random_state=sd, **CFG.GBM)
        m.fit(z['X'], z['Y'], sample_weight=W)
        RANKERS.append(m)

def rank_proba(X):
    return np.mean([m.predict_proba(X)[:, 1] for m in RANKERS], axis=0)

rep, rep_key, rep_nm, rep_is_pos = build_rep_polarity(L)

def run_pipeline_eval(engine_type='phase1'):
    """Evaluates full pipeline on val.parquet with leakage protection."""
    rows = []
    print(f"\n[INFO] Starting Full Pipeline Benchmark: Engine = {engine_type.upper()}...")
    t_start = time.time()

    for gi, (mid, sub) in enumerate(val_mols):
        nms = sub.nm.values[np.isfinite(sub.nm.values)]
        smis = []
        if len(nms):
            raw_target = float(np.median(nms))
            if engine_type == 'phase1':
                target = raw_target * (1.0 - 1.45e-6)
            else:
                target = raw_target

            specs = [(r.ms2_mzs, r.ms2_normalized_intensities) for r in sub.itertuples()]
            specs_with_modes = [(r.ms2_mzs, r.ms2_normalized_intensities, str(r.ionization_mode).lower() == 'positive')
                                for r in sub.itertuples()]
            br, bk = BAN[mid]

            # Channel 1: Direct Library Similarity (with leakage dropping)
            cand_lib = lib_window(L, target, target * CFG.PPM_WIN_BASE / 1e6)
            cand_lib = _drop(cand_lib, br, bk, L['ik'])
            lib_hits = {}
            if len(cand_lib):
                for mz, it in specs:
                    qm, qp = clean_spectrum(mz, it)
                    if len(qm) == 0: continue
                    sc = search(qm, qp, cand_lib, L['off'], L['mz'], L['it'],
                                CFG.MZ_TOL, CFG.INT_FLOOR, CFG.MAX_PEAKS, CFG.INT_POWER, CFG.ENT_WEIGHT)
                    for c, s in zip(cand_lib, sc):
                        k = L['ik'][c]
                        if s > lib_hits.get(k, -1.0): lib_hits[k] = float(s)

            # Channel 2: Analog Matching (with leakage dropping)
            if engine_type == 'phase1':
                analogs = analog_sim_polarity(L, specs_with_modes, target, rep, rep_key, rep_nm, rep_is_pos, br, bk)
            else:
                analogs = analog_sim_baseline(L, specs, target, rep, rep_key, rep_nm, br, bk)

            # Candidate Window Selection
            if engine_type == 'phase1':
                cand = pool.window(target, CFG.PPM_EXPAND[0])
                if len(cand) < CFG.TOPN:
                    for ppm_tier in CFG.PPM_EXPAND[1:]:
                        cand_tier = pool.window(target, ppm_tier)
                        if len(cand_tier) >= CFG.TOPN:
                            cand = cand_tier
                            break
                        if len(cand_tier) > len(cand): cand = cand_tier
            else:
                cand = pool.window(target, CFG.PPM_WIN_BASE)
                if len(cand) == 0:
                    cand = pool.window(target, CFG.PPM_FALLBACK)

            if len(cand):
                cfp = pool.fps(cand)
                lv  = np.array([lib_hits.get(pool.keys[c], 0.0) for c in cand], np.float32)
                ids, sims = [], []
                for k, s in analogs:
                    i = pool.k2i.get(k, -1)
                    if i >= 0: ids.append(i); sims.append(s)
                afp = pool.fps(np.array(ids)) if ids else None

                ion_mode = float(np.mean([1.0 if m == 'positive' else -1.0 for m in sub.ionization_mode]))
                fsc = frag_scores([pool.smiles[c] for c in cand], specs, ion_mode)
                zlog = model_logits(sub)

                X = rank_features(cfp, lv, afp, np.array(sims, np.float32), zlog, fsc)[:, :NFEAT]
                p = rank_proba(X)
                order = np.argsort(-p)

                if engine_type == 'phase1':
                    seen_keys = set()
                    for idx in order:
                        c_idx = cand[idx]
                        k = pool.keys[c_idx]
                        if k not in seen_keys and k is not None and len(str(k)) >= 14:
                            seen_keys.add(k)
                            smis.append(pool.smiles[c_idx])
                            if len(smis) == CFG.TOPN: break

                    if len(smis) < CFG.TOPN:
                        wide_cand = pool.window(target, 120.0)
                        if len(wide_cand):
                            mass_diff = np.abs(pool.mass[wide_cand] - target)
                            nearest_order = np.argsort(mass_diff)
                            for c_idx in wide_cand[nearest_order]:
                                k = pool.keys[c_idx]
                                if k not in seen_keys and k is not None and len(str(k)) >= 14:
                                    seen_keys.add(k)
                                    smis.append(pool.smiles[c_idx])
                                    if len(smis) == CFG.TOPN: break
                else:
                    smis = [pool.smiles[cand[i]] for i in order[:CFG.TOPN]]

        while len(smis) < CFG.TOPN:
            smis.append('CCO')

        rows.append((mid, ';'.join(smis[:CFG.TOPN])))
        if gi % 200 == 0 or gi == len(val_mols) - 1:
            print(f"  [{engine_type.upper()}] Processed {gi+1}/{len(val_mols)} molecules ({time.time()-t_start:.1f}s)", flush=True)

    return pd.DataFrame(rows, columns=['molecule_id', 'smiles'])

# Evaluation metric function
def mrr_at_25(sub_df, key_list, pool, k=25):
    s2k = {s: kk for s, kk in zip(pool.smiles, pool.keys)}
    ans = {r['molecule_id']: (r['inchikey14'], r['cohort']) for r in key_list}
    per = {}
    for mid, smi in zip(sub_df.molecule_id, sub_df.smiles):
        if mid not in ans: continue
        truth, coh = ans[mid]
        rank, seen = 0, set()
        for t in str(smi).split(';')[:k]:
            kk = s2k.get(t)
            if kk is None or kk in seen: continue
            seen.add(kk); rank += 1
            if kk == truth:
                per.setdefault(coh, []).append(1.0 / rank); break
        else:
            per.setdefault(coh, []).append(0.0)

    rows, allv = [], []
    for c, v in sorted(per.items()):
        allv += v
        rows.append(dict(cohort=c, n=len(v), MRR=np.mean(v),
                         top1=np.mean([x == 1.0 for x in v]),
                         hit25=np.mean([x > 0 for x in v])))
    rows.append(dict(cohort='ALL', n=len(allv), MRR=np.mean(allv),
                     top1=np.mean([x == 1.0 for x in allv]),
                     hit25=np.mean([x > 0 for x in allv])))
    return pd.DataFrame(rows).round(4)


In [ ]:
# ===================================================================================
#  Run Side-by-Side Validation & Display Comparison Scorecard
# ===================================================================================
sub_base   = run_pipeline_eval(engine_type='baseline')
sub_phase1 = run_pipeline_eval(engine_type='phase1')

score_base   = mrr_at_25(sub_base, key_list, pool)
score_phase1 = mrr_at_25(sub_phase1, key_list, pool)

# Assemble Comparative Scorecard
scorecard = pd.DataFrame({
    'Cohort': score_base['cohort'],
    'N': score_base['n'],
    'Base MRR': score_base['MRR'],
    'Phase 1 MRR': score_phase1['MRR'],
    'Delta MRR': (score_phase1['MRR'] - score_base['MRR']).round(4),
    'Base Top-1': score_base['top1'],
    'Phase 1 Top-1': score_phase1['top1'],
    'Base Hit@25': score_base['hit25'],
    'Phase 1 Hit@25': score_phase1['hit25'],
})

print("\n===================================================================================")
print("               🏆 CASMI 2026 VALIDATION COMPARATIVE SCORECARD                     ")
print("===================================================================================")
print(scorecard.to_string(index=False))
print("===================================================================================")
